In [1]:
k = 16
dmodel = 3584
print(f'dmodel: {dmodel}')
n_layers = 16
tt_params = 12 * n_layers * (dmodel ** 2)
emb_params = 50_000 * dmodel
model_params = emb_params + tt_params

dmodel: 3584


In [2]:
batch_size = 512
seq_len = 512
steps = 20_000

tokens = batch_size * seq_len * steps
print(tokens / 1e9)

print(f'tokens/params ratio:{tokens/model_params:.2f}')

5.24288
tokens/params ratio:1.98


In [3]:
gpus = 4
time = 12 * (60 ** 2)
gpu_flops = 835 * 1e12  #H100
# gpu_flops = 312 * 1e12  #A100

In [4]:
flops = model_params * tokens
mfu = flops / (time * gpus * gpu_flops)
print(f'MFU: {mfu:.3f}')

MFU: 0.096


In [5]:
current_steps = 112
current_time = 4 * 60
total_steps = 20_000
time_total = (total_steps / current_steps) * current_time
print(f'time left: {time_total / (60 **2 ):.2f}')

time left: 11.90


In [6]:
def calc_grid_flops(n_lrs_per_dmodel: int, dmodels: list, n_layers: int, tokens: int, n_plots: int):
    total_flops = 0
    flops_list = []
    for dmodel in dmodels:
        params = 12 * (dmodel ** 2) * n_layers
        flops = tokens * params * n_lrs_per_dmodel
        flops_list.append(flops)
        total_flops += flops
    
    return total_flops * n_plots

In [7]:
grid_dense = {
    "n_lrs_per_dmodel": 5,
    "dmodels": [128, 256, 512, 768, 1024, 1536],
    "n_layers": 24,
    "tokens": 16 * 1e9,
    "n_plots": 2
}

grid_moe_new = {
    "n_lrs_per_dmodel": 5,
    "dmodels": [128, 256, 512, 768],
    "n_layers": 16,
    "tokens": 5 * 1e9,
    "n_plots": 1,
}

grid_moe_new_helios = {
    "n_lrs_per_dmodel": 5,
    "dmodels": [2048, 3584],
    "n_layers": 16,
    "tokens": 5 * 1e9,
    "n_plots": 3,
}

current_128 = {
    "n_lrs_per_dmodel": 5,
    "dmodels": [128],
    "n_layers": 24,
    "tokens": 16 * 1e9,
    "n_plots": 3,
}
current_256 = {
    "n_lrs_per_dmodel": 5,
    "dmodels": [256],
    "n_layers": 24,
    "tokens": 16 * 1e9,
    "n_plots": 3,
}
current_512 = {
    "n_lrs_per_dmodel": 4,
    "dmodels": [512],
    "n_layers": 24,
    "tokens": 16 * 1e9,
    "n_plots": 1,
}
current_1024 = {
    "n_lrs_per_dmodel": 1,
    "dmodels": [1024],
    "n_layers": 24,
    "tokens": 16 * 1e9,
    "n_plots": 1,
}


dense_flops = calc_grid_flops(**grid_dense)
print(f"{dense_flops / 1e18} * 1e18")

dmodels = [128, 256, 512, 768, 1024, 1536, 2048, 3584]
for dm in dmodels:
    grid_moe_new["dmodels"] = [dm]
    flops_dm = calc_grid_flops(**grid_moe_new)
    print(f"dmodel: {dm}\tflops:{flops_dm/1e18:.2f} * 1e18")


200.0683008 * 1e18
dmodel: 128	flops:0.08 * 1e18
dmodel: 256	flops:0.31 * 1e18
dmodel: 512	flops:1.26 * 1e18
dmodel: 768	flops:2.83 * 1e18
dmodel: 1024	flops:5.03 * 1e18
dmodel: 1536	flops:11.32 * 1e18
dmodel: 2048	flops:20.13 * 1e18
dmodel: 3584	flops:61.66 * 1e18


In [8]:
h100_flops = 835 * 1e12  #H100
a100_flops = 312 * 1e12  #A100
h100_flops_per_day = 24 * 60 * 60 * h100_flops
a100_flops_per_day = 24 * 60 * 60 * a100_flops
mfu = 0.1
print(f"{a100_flops_per_day * 8 * mfu / 1e18} * 1e18")
print(f"{h100_flops_per_day * 8 * mfu / 1e18} * 1e18")


21.56544 * 1e18
57.7152 * 1e18


In [9]:
1536 + 2048

3584